# Nocturne Aegis Cel Candidate Generator v2

Corrected notebook for `OnomaAIResearch/Illustrious-xl-early-release-v0`.

Goal: generate **80 candidate images** from **10 prompts x 8 seeds** for later LoRA dataset selection.

Main fixes vs v1:
- tag-style prompt format instead of long prose
- strong full-color / finished-image quality tags
- negative prompt blocks sketch, monochrome, grayscale, rough lineart, unfinished images
- SDXL-specific `StableDiffusionXLPipeline`
- separate `prompt` and `prompt_2`
- optional `clip_skip=2`
- per-prompt aspect ratios

In [ ]:
# Install dependencies in Colab
!pip install -q -U diffusers transformers accelerate safetensors pillow pandas huggingface_hub

In [ ]:
import os, json, gc, inspect, shutil
from pathlib import Path
from datetime import datetime

import torch
import pandas as pd
from PIL import Image, ImageDraw
from diffusers import StableDiffusionXLPipeline, EulerAncestralDiscreteScheduler, DPMSolverMultistepScheduler

print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('CUDA capability:', torch.cuda.get_device_capability(0))
    torch.backends.cuda.matmul.allow_tf32 = True

## Optional: mount Google Drive

Use this if you want the generated outputs to persist after the Colab runtime disconnects.

In [ ]:
# Optional: uncomment to save directly to Google Drive
# from google.colab import drive
# drive.mount('/content/drive')

## Optional: Hugging Face login

Run only if the model download fails due to access/authentication.

In [ ]:
# Optional: uncomment if authentication is required
# from huggingface_hub import notebook_login
# notebook_login()

In [ ]:
MODEL_ID = 'OnomaAIResearch/Illustrious-xl-early-release-v0'
PROJECT_NAME = 'nocturne_aegis_candidates_v2'

# Change to /content/drive/MyDrive/nocturne_aegis_outputs if using Drive.
OUTPUT_ROOT = '/content/outputs'

# 10 prompts x 8 images = 80 candidates.
NUM_IMAGES_PER_PROMPT = 8
BASE_SEED = 24052026

# Defaults used when a prompt does not specify width/height.
DEFAULT_WIDTH = 1024
DEFAULT_HEIGHT = 1024

# Better first-run values for Illustrious/Kohaku-family SDXL models.
NUM_INFERENCE_STEPS = 40
GUIDANCE_SCALE = 6.5
CLIP_SKIP = 2

# T4: True. A100/L4: False is faster.
USE_CPU_OFFLOAD = True

# Options: 'euler_a', 'dpmpp_2m_karras', 'model_default'
SCHEDULER = 'euler_a'

# Save paths
output_dir = Path(OUTPUT_ROOT) / PROJECT_NAME
images_dir = output_dir / 'images'
images_dir.mkdir(parents=True, exist_ok=True)
print('Output directory:', output_dir)

In [ ]:
# Illustrious/Kohaku-style prompt engineering.
# Keep this concise and tag-like. Very long prose often gets clipped or interpreted poorly.

QUALITY_PREFIX = (
    'masterpiece, best quality, very aesthetic, absurdres, high resolution, '
    'official art, finished full-color anime illustration, polished digital illustration, '
    'clean final render, sharp focus'
)

NOCTURNE_STYLE_TAGS = (
    'Nocturne Aegis Cel, 1990s anime cel shading, modern volumetric digital painting, '
    'elegant elongated proportions, 8-head figure ratio, fragile anatomical tension, '
    'sharp geometric face, angular jaw, melancholic expression, detailed glass-like eyes, '
    'geometric irises, large catchlights, clean tapered linework, varied line weight, '
    'bold ink contours on armor and machinery, jewel tone palette, blackened indigo, violet, '
    'magenta, sapphire, antique gold, pale silver, saturated colors, controlled gradient color, '
    'layered cel shadows, hard shadow terminator, ambient occlusion, deep contact shadows, '
    'extreme chiaroscuro, dramatic key light, soft fill light, strong rim light, '
    'iridescent metal, sharp specular highlights, burnished chrome, matte velvet cloth, '
    'cinematic isolation, ethereal fantasy atmosphere, clear silhouette, intentional negative space'
)

# Critical: v1 missed these negatives, so the model drifted into rough sketch / monochrome outputs.
NEGATIVE_PROMPT = (
    'worst quality, low quality, normal quality, lowres, bad anatomy, bad hands, '
    'extra fingers, missing fingers, malformed fingers, broken wrist, distorted face, '
    'ugly, deformed, disfigured, blurry, out of focus, jpeg artifacts, text, watermark, logo, '
    'signature, cropped hands, cropped feet, duplicate limbs, fused armor, messy details, '
    'rough sketch, sketch, pencil sketch, graphite, charcoal drawing, monochrome, grayscale, '
    'lineart only, outline only, unfinished, draft, colorless, washed out, flat lighting, '
    'muddy colors, overexposed, underexposed, photorealistic, 3d render, plastic skin'
)

# A separate SDXL prompt_2 helps preserve style because SDXL uses a second text encoder.
PROMPT_2_STYLE = (
    'high-end finished anime illustration, full color, clean cel shaded line art with modern digital painting, '
    'jewel-toned cinematic fantasy, deep ambient occlusion, dramatic chiaroscuro, glass eyes, '
    'iridescent armor, matte cloth, refined face and hands, clean silhouette, polished final render'
)

NEGATIVE_PROMPT_2 = (
    'sketch, monochrome, grayscale, rough drawing, unfinished, low quality, bad anatomy, bad hands, watermark, text'
)

print('Quality prefix chars:', len(QUALITY_PREFIX))
print('Style tags chars:', len(NOCTURNE_STYLE_TAGS))
print('Negative chars:', len(NEGATIVE_PROMPT))

In [ ]:
# Load prompts. In Colab, upload prompts_illustrious_optimized.json to /content first.
# If you do not upload it, the notebook uses the built-in prompts below.

DEFAULT_PROMPTS = json.loads(r'''[{"id": "nacel_001_sky_pilot", "content_prompt": "adult sky pilot, standing on broken launch bridge, cracked helmet in one hand, long coat panels blowing in wind, distant gothic starships, ruined skyline, low angle, full body", "caption": "nacel_v1, an adult sky pilot standing on a broken launch bridge holding a cracked helmet, long coat panels blowing in the wind, distant gothic starships in a ruined skyline, low-angle full-body composition", "width": 832, "height": 1216}, {"id": "nacel_002_cockpit_swordsman", "content_prompt": "adult male swordsman, short silver hair, seated inside damaged cockpit, armored coat open at collar, sheathed blade in hand, instrument lights behind him, three-quarter bust", "caption": "nacel_v1, an adult male swordsman with short silver hair seated inside a damaged cockpit, wearing an armored coat and holding a sheathed blade, three-quarter bust composition", "width": 1024, "height": 1024}, {"id": "nacel_003_flooded_cathedral_knight", "content_prompt": "masked androgynous knight, standing in flooded cathedral corridor, reflective black water, narrow vertical windows, long ribbons and cables around body, centered full body", "caption": "nacel_v1, a masked androgynous knight standing in a flooded cathedral corridor with reflective water, long ribbons and cables around the body, centered full-body composition", "width": 832, "height": 1216}, {"id": "nacel_004_gauntlet_mechanic", "content_prompt": "adult woman with cropped dark hair, repairing biomechanical gauntlet on workbench, tools and small metal parts, focused downward expression, hands visible, dark workshop, close portrait", "caption": "nacel_v1, an adult woman with cropped dark hair repairing a biomechanical gauntlet on a workbench, tools and metal parts around her, close portrait with hands visible", "width": 1024, "height": 1024}, {"id": "nacel_005_moon_engine_priest", "content_prompt": "ceremonial armored priest, standing before suspended moon engine, layered black robes, pale metal shoulder guards, one hand raised in restrained ritual gesture, monumental vertical architecture, wide shot", "caption": "nacel_v1, a ceremonial armored priest standing before a suspended moon engine, wearing layered black robes and pale metal shoulder guards, one hand raised, wide environmental shot", "width": 1216, "height": 832}, {"id": "nacel_006_winged_mech", "content_prompt": "winged humanoid machine, kneeling on cathedral bridge, folded mechanical wings, visible joints, plated limbs, no human rider, distant city ruins below, high angle, dramatic wide shot", "caption": "nacel_v1, a winged humanoid machine kneeling on a cathedral bridge with folded mechanical wings and visible joints, distant city ruins below, high-angle wide shot", "width": 1216, "height": 832}, {"id": "nacel_007_observatory_archer", "content_prompt": "adult archer, layered traveling clothes, light armor, standing in ruined observatory, bow lowered at side, star maps and broken brass instruments, calm profile, full body", "caption": "nacel_v1, an adult archer wearing layered traveling clothes and light armor standing in a ruined observatory, holding a lowered bow, calm profile full-body pose", "width": 832, "height": 1216}, {"id": "nacel_008_helmet_study", "content_prompt": "ornate helmet with glass visor, resting on dark velvet fabric, antique metal trim, fine scratches, broken feather crest, no person, object study, shallow background", "caption": "nacel_v1, an ornate helmet with a glass visor resting on dark velvet fabric, antique metal trim, fine scratches, broken crest, object study composition", "width": 1024, "height": 1024}, {"id": "nacel_009_celestial_wolf", "content_prompt": "armored celestial wolf, standing on cracked stone platform, mechanical halo, layered metal plates, long ribbon-like energy trails, moonlit ruins behind it, creature full body", "caption": "nacel_v1, an armored celestial wolf standing on a cracked stone platform with a mechanical halo and layered metal plates, ribbon-like energy trails, moonlit ruins behind it, creature full-body composition", "width": 1024, "height": 1024}, {"id": "nacel_010_gothic_starfighter", "content_prompt": "gothic starfighter vehicle, parked on rain-soaked launch rail, folded blade-like wings, cockpit canopy glowing, no pilot visible, vertical city towers in background, wide object and vehicle study", "caption": "nacel_v1, a gothic starfighter parked on a rain-soaked launch rail with folded blade-like wings and a glowing cockpit canopy, vertical city towers in the background, wide vehicle study", "width": 1216, "height": 832}]''')

prompt_paths = [
    Path('/content/prompts_illustrious_optimized.json'),
    Path('/content/prompts_illustrorious_optimized.json'),
    Path('/content/prompts.json'),
]

PROMPTS = None
for p in prompt_paths:
    if p.exists():
        PROMPTS = json.loads(p.read_text(encoding='utf-8'))
        print(f'Loaded {len(PROMPTS)} prompts from {p}')
        break
if PROMPTS is None:
    PROMPTS = DEFAULT_PROMPTS
    print(f'Using {len(PROMPTS)} built-in prompts')

PROMPTS[0]

In [ ]:
# Load pipeline. Use float16 on GPU; bfloat16 can work on A100 but float16 is more predictable across Colab GPUs.
DTYPE = torch.float16 if torch.cuda.is_available() else torch.float32
print('Using dtype:', DTYPE)

pipe = StableDiffusionXLPipeline.from_pretrained(
    MODEL_ID,
    torch_dtype=DTYPE,
    use_safetensors=True,
    add_watermarker=False,
)

if SCHEDULER == 'euler_a':
    pipe.scheduler = EulerAncestralDiscreteScheduler.from_config(pipe.scheduler.config)
elif SCHEDULER == 'dpmpp_2m_karras':
    pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config, use_karras_sigmas=True)

pipe.enable_vae_slicing()
try:
    pipe.enable_vae_tiling()
except Exception as e:
    print('VAE tiling not enabled:', e)

if torch.cuda.is_available():
    if USE_CPU_OFFLOAD:
        pipe.enable_model_cpu_offload()
        print('Model CPU offload enabled')
    else:
        pipe.to('cuda')
        print('Pipeline moved to CUDA')
else:
    print('CUDA unavailable. CPU generation will be very slow.')

# Detect if this Diffusers version supports clip_skip.
CALL_SUPPORTS_CLIP_SKIP = 'clip_skip' in inspect.signature(pipe.__call__).parameters
print('clip_skip supported:', CALL_SUPPORTS_CLIP_SKIP)
print('Scheduler:', pipe.scheduler.__class__.__name__)

In [ ]:
def build_prompt(content_prompt: str) -> str:
    # Tag-style prompt with quality first, then content, then stable style tags.
    return f'{QUALITY_PREFIX}, {content_prompt}, {NOCTURNE_STYLE_TAGS}'

def generation_kwargs(width: int, height: int, generator):
    kwargs = dict(
        negative_prompt=NEGATIVE_PROMPT,
        prompt_2=PROMPT_2_STYLE,
        negative_prompt_2=NEGATIVE_PROMPT_2,
        width=width,
        height=height,
        num_inference_steps=NUM_INFERENCE_STEPS,
        guidance_scale=GUIDANCE_SCALE,
        generator=generator,
        original_size=(width, height),
        target_size=(width, height),
        crops_coords_top_left=(0, 0),
    )
    if CALL_SUPPORTS_CLIP_SKIP:
        kwargs['clip_skip'] = CLIP_SKIP
    return kwargs

def make_contact_sheet(image_paths, out_path, thumb_size=(256, 256), cols=4):
    image_paths = [Path(p) for p in image_paths]
    if not image_paths:
        return None
    rows = (len(image_paths) + cols - 1) // cols
    cell_w, cell_h = thumb_size[0], thumb_size[1] + 40
    sheet = Image.new('RGB', (cols * cell_w, rows * cell_h), 'white')
    draw = ImageDraw.Draw(sheet)
    for i, p in enumerate(image_paths):
        img = Image.open(p).convert('RGB')
        img.thumbnail(thumb_size)
        x = (i % cols) * cell_w + (cell_w - img.width) // 2
        y = (i // cols) * cell_h
        sheet.paste(img, (x, y))
        draw.text(((i % cols) * cell_w + 5, y + thumb_size[1] + 4), p.name[:34], fill=(0, 0, 0))
    sheet.save(out_path, quality=92)
    return out_path

## Smoke test

Run this before the full 80-image batch. It generates 2 images from prompt 1.

If these still look like rough sketches, check that the negative prompt includes `sketch`, `monochrome`, `grayscale`, and `unfinished`, then try `GUIDANCE_SCALE = 7.0`.

In [ ]:
SMOKE_TEST_COUNT = 2
smoke_paths = []
item = PROMPTS[0]
content_prompt = item['content_prompt'].strip()
width = int(item.get('width', DEFAULT_WIDTH))
height = int(item.get('height', DEFAULT_HEIGHT))
full_prompt = build_prompt(content_prompt)

print('SMOKE PROMPT:')
print(full_prompt[:1200])
print('
WIDTH/HEIGHT:', width, height)

for i in range(SMOKE_TEST_COUNT):
    seed = BASE_SEED + 900000 + i
    generator_device = 'cuda' if torch.cuda.is_available() else 'cpu'
    generator = torch.Generator(device=generator_device).manual_seed(seed)
    out_path = images_dir / f'smoke_{item["id"]}_seed_{seed}.png'
    with torch.inference_mode():
        result = pipe(prompt=full_prompt, **generation_kwargs(width, height, generator))
    result.images[0].save(out_path)
    smoke_paths.append(out_path)
    print('Saved:', out_path)
    del result
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

make_contact_sheet(smoke_paths, output_dir / 'smoke_contact_sheet.jpg', cols=2)
Image.open(output_dir / 'smoke_contact_sheet.jpg')

## Full 80-image candidate run

Run after the smoke test looks acceptable.

In [ ]:
metadata_rows = []
generated_paths = []
run_started_at = datetime.utcnow().isoformat()

for prompt_index, item in enumerate(PROMPTS, start=1):
    prompt_id = item['id']
    content_prompt = item['content_prompt'].strip()
    caption = item['caption'].strip()
    width = int(item.get('width', DEFAULT_WIDTH))
    height = int(item.get('height', DEFAULT_HEIGHT))
    full_prompt = build_prompt(content_prompt)

    print(f'
=== {prompt_index:02d}/{len(PROMPTS)} | {prompt_id} | {width}x{height} ===')

    for variation_index in range(1, NUM_IMAGES_PER_PROMPT + 1):
        seed = BASE_SEED + (prompt_index * 1000) + variation_index
        generator_device = 'cuda' if torch.cuda.is_available() else 'cpu'
        generator = torch.Generator(device=generator_device).manual_seed(seed)

        filename_base = f'{prompt_id}_seed_{seed}_{variation_index:02d}'
        image_path = images_dir / f'{filename_base}.png'
        caption_path = images_dir / f'{filename_base}.txt'

        if image_path.exists():
            print('Skipping existing:', image_path.name)
            generated_paths.append(image_path)
            continue

        with torch.inference_mode():
            result = pipe(prompt=full_prompt, **generation_kwargs(width, height, generator))

        image = result.images[0]
        image.save(image_path)
        caption_path.write_text(caption + '
', encoding='utf-8')
        generated_paths.append(image_path)

        metadata_rows.append({
            'filename': image_path.name,
            'caption_filename': caption_path.name,
            'prompt_id': prompt_id,
            'prompt_index': prompt_index,
            'variation_index': variation_index,
            'seed': seed,
            'model_id': MODEL_ID,
            'scheduler': pipe.scheduler.__class__.__name__,
            'width': width,
            'height': height,
            'num_inference_steps': NUM_INFERENCE_STEPS,
            'guidance_scale': GUIDANCE_SCALE,
            'clip_skip': CLIP_SKIP if CALL_SUPPORTS_CLIP_SKIP else None,
            'content_prompt': content_prompt,
            'full_prompt': full_prompt,
            'prompt_2': PROMPT_2_STYLE,
            'caption': caption,
            'negative_prompt': NEGATIVE_PROMPT,
            'negative_prompt_2': NEGATIVE_PROMPT_2,
            'generated_at_utc': datetime.utcnow().isoformat(),
        })

        print('Saved:', image_path.name)
        del result, image
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

metadata_df = pd.DataFrame(metadata_rows)
metadata_csv_path = output_dir / 'metadata.csv'
metadata_jsonl_path = output_dir / 'metadata.jsonl'
metadata_df.to_csv(metadata_csv_path, index=False)
metadata_df.to_json(metadata_jsonl_path, orient='records', lines=True, force_ascii=False)

contact_sheet_path = output_dir / 'contact_sheet.jpg'
make_contact_sheet(sorted(images_dir.glob('*.png')), contact_sheet_path, cols=5)

archive_path = shutil.make_archive(str(output_dir / 'output_archive'), 'zip', output_dir)

print('
Done.')
print('Newly generated this run:', len(metadata_df))
print('All image files in folder:', len(list(images_dir.glob('*.png'))))
print('Metadata:', metadata_csv_path)
print('Contact sheet:', contact_sheet_path)
print('Archive:', archive_path)

In [ ]:
# Display final contact sheet
contact_sheet_path = output_dir / 'contact_sheet.jpg'
if contact_sheet_path.exists():
    display(Image.open(contact_sheet_path))
else:
    print('No contact sheet found yet.')